# Linear Regression Consulting Project

#### Task
Congratulations! You've been contracted by Hyundai Heavy Industries to help them build a predictive model for some ships. [Hyundai Heavy Industries](http://www.hyundai.eu/en) is one of the world's largest ship manufacturing companies and builds cruise liners.

You've been flown to their headquarters in Ulsan, South Korea to help them give accurate estimates of how many crew members a ship will require.

They are currently building new ships for some customers and want you to create a model and use it to predict how many crew members the ships will need.

Here is what the data looks like so far:

    Description: Measurements of ship size, capacity, crew, and age for 158 cruise
    ships.


    Variables/Columns
    Ship Name     1-20
    Cruise Line   21-40
    Age (as of 2013)   46-48
    Tonnage (1000s of tons)   50-56
    passengers (100s)   58-64
    Length (100s of feet)  66-72
    Cabins  (100s)   74-80
    Passenger Density   82-88
    Crew  (100s)   90-96
    
It is saved in a csv file for you called "cruise_ship_info.csv". Your job is to create a regression model that will help predict how many crew members will be needed for future ships. The client also mentioned that they have found that particular cruise lines will differ in acceptable crew counts, so it is most likely an important feature to include in your analysis! 

Once you've created the model and tested it for a quick check on how well you can expect it to perform, make sure you take a look at why it performs so well!

#### Import needed libraries

In [ ]:
import pandas as pd
from pyspark.sql.functions import col, count, round as spark_round, row_number, array, lit
from pyspark.sql.window import Window
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

#### Start a simple Spark Session

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("CruiseShipCrew").getOrCreate()

#### Load Cruise Ship Data

We'll use PySpark to load the 'cruise_ship_info.csv' dataset and inspect its structure.

In [0]:
path = 'data/cruise_ship_info.csv'
df = spark.read.csv(path, header=True, inferSchema=True)
df.show(5)

+-----------+-----------+---+------------------+----------+------+------+-----------------+----+
|  Ship_name|Cruise_line|Age|           Tonnage|passengers|length|cabins|passenger_density|crew|
+-----------+-----------+---+------------------+----------+------+------+-----------------+----+
|    Journey|    Azamara|  6|30.276999999999997|      6.94|  5.94|  3.55|            42.64|3.55|
|      Quest|    Azamara|  6|30.276999999999997|      6.94|  5.94|  3.55|            42.64|3.55|
|Celebration|   Carnival| 26|            47.262|     14.86|  7.22|  7.43|             31.8| 6.7|
|   Conquest|   Carnival| 11|             110.0|     29.74|  9.53| 14.88|            36.99|19.1|
|    Destiny|   Carnival| 17|           101.353|     26.42|  8.92| 13.21|            38.36|10.0|
+-----------+-----------+---+------------------+----------+------+------+-----------------+----+
only showing top 5 rows


#### Exploratory Data Analysis (EDA)

We'll explore the data for missing values, distributions, and relationships between variables. This step helps us understand the features and target variable ('Crew').

In [0]:
# Count missing values per column
df.select([count(col(c)).alias(c) for c in df.columns]).show()

# Describe quantitative columns
numeric_cols = ['Age', 'Tonnage', 'passengers', 'length', 'cabins', 'passenger_density', 'crew']
desc = df.describe(numeric_cols)
float_cols = [c for c in desc.columns if c != 'summary']
for c in float_cols:
    desc = desc.withColumn(c, spark_round(col(c), 3))
desc.show()

+---------+-----------+---+-------+----------+------+------+-----------------+----+
|Ship_name|Cruise_line|Age|Tonnage|passengers|length|cabins|passenger_density|crew|
+---------+-----------+---+-------+----------+------+------+-----------------+----+
|      158|        158|158|    158|       158|   158|   158|              158| 158|
+---------+-----------+---+-------+----------+------+------+-----------------+----+

+-------+-----+-------+----------+------+------+-----------------+-----+
|summary|  Age|Tonnage|passengers|length|cabins|passenger_density| crew|
+-------+-----+-------+----------+------+------+-----------------+-----+
|  count|158.0|  158.0|     158.0| 158.0| 158.0|            158.0|158.0|
|   mean|15.69| 71.285|    18.457| 8.131|  8.83|           39.901|7.794|
| stddev|7.616|  37.23|     9.677| 1.793| 4.471|            8.639|3.503|
|    min|  4.0|  2.329|      0.66|  2.79|  0.33|             17.7| 0.59|
|    max| 48.0|  220.0|      54.0| 11.82|  27.0|            71.43| 2

In [0]:
# Basic correlation analysis
numeric_cols = ['Age','Tonnage','passengers','length','cabins','passenger_density','crew']
correlations = {}
for colname in numeric_cols[:-1]:
    corr = df.stat.corr(colname, 'crew')
    correlations[colname] = corr
print('Correlation with crew:')
for k, v in correlations.items():
    print(f'{k}: {v:.3f}')

Correlation with crew:
Age: -0.531
Tonnage: 0.928
passengers: 0.915
length: 0.896
cabins: 0.951
passenger_density: -0.156


#### Feature Engineering & Data Preparation

We'll encode categorical features and assemble the input features for model training.

In [0]:
# Add dummy partition column to avoid Window warning
cruise_lines_df = df.select('Cruise_line').distinct().withColumn('dummy', lit(1))
windowSpec = Window.partitionBy('dummy').orderBy('Cruise_line')
cruise_lines_mapping = cruise_lines_df.withColumn(
    'Cruise_Line_Index', row_number().over(windowSpec) - 1
).drop('dummy')

# Join mapping back to original DataFrame
df_encoded = df.join(cruise_lines_mapping, on='Cruise_line', how='left')

feature_cols = ['Age','Tonnage','passengers','length','cabins','passenger_density','Cruise_Line_Index']
df_final = df_encoded.withColumn('features', array(*[col(c) for c in feature_cols]))
df_final.select('features', 'crew').show(5)

+--------------------+----+
|            features|crew|
+--------------------+----+
|[6.0, 30.27699999...|3.55|
|[6.0, 30.27699999...|3.55|
|[26.0, 47.262, 14...| 6.7|
|[11.0, 110.0, 29....|19.1|
|[17.0, 101.353, 2...|10.0|
+--------------------+----+
only showing top 5 rows


#### Train/Test Split

We'll split the data for training and evaluation.

In [0]:
train_data, test_data = df_final.randomSplit([0.7, 0.3], seed=42)
print(f'Train count: {train_data.count()}, Test count: {test_data.count()}')

Train count: 115, Test count: 43


#### Linear Regression Modeling

We'll build a pipeline to fit a linear regression model to predict Crew size based on ship features and cruise line.

In [0]:
# Convert to pandas DataFrame
pdf = df_final.select(
    'Age','Tonnage','passengers','length','cabins','passenger_density','Cruise_Line_Index','crew').toPandas()

X = pdf[['Age','Tonnage','passengers','length','cabins','passenger_density','Cruise_Line_Index']]
y = pdf['crew']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

#### Model Evaluation

Evaluate performance (RMSE, R2) and check prediction quality.

In [0]:
# Make predictions
preds = lr.predict(X_test)

# Performance
rmse = mean_squared_error(y_test, preds, squared=False)
r2 = r2_score(y_test, preds)
print(f'RMSE: {rmse:.2f}, R2: {r2:.2f}')

# Show a few predictions
result = pd.DataFrame({
    'Actual': y_test.values,
    'Prediction': preds
})
print(result.head())

RMSE: 0.88, R2: 0.94
   Actual  Prediction
0    7.20    7.912765
1    6.36    6.480988
2   21.00   20.636152
3    0.88    0.858161
4    3.50    3.627153


#### Feature Importance & Insights

Review coefficients and feature importance to understand which features drive Crew predictions.

In [0]:
# Feature importance (coefficients)
features = ['Age','Tonnage','passengers','length','cabins','passenger_density','Cruise_Line_Index']
coefs = lr.coef_
intercept = lr.intercept_
for f, c in zip(features, coefs):
    print(f'{f}: {c:.3f}')
print(f'Intercept: {intercept:.3f}')

Age: -0.005
Tonnage: 0.009
passengers: -0.126
length: 0.347
cabins: 0.816
passenger_density: 0.015
Cruise_Line_Index: -0.036
Intercept: -0.779


#### Results

The regression model predicts cruise ship crew size with high accuracy, achieving an R² of 0.94 and RMSE of 0.88. Cabins, ship length, and tonnage are the most influential features, while cruise line effects are modest. Sample predictions closely match actual crew values, confirming the model's reliability.